In [2]:
import numpy as np
from pathlib import Path

# ============================================================
# Assignment configuration: Topic 6 / FSS
# ============================================================
NUM_BANKS = 4
BANK_SIZE = 8
ROW_SIZE = NUM_BANKS * BANK_SIZE
KEEP_PER_BANK = 4
SPARSITY = 0.50

OUT = Path("/content/data")
OUT.mkdir(parents=True, exist_ok=True)


# Reproducible example data. Replace these arrays with one real
# 32-element weight row and one 32-element activation vector.
rng = np.random.default_rng(17)
weights = rng.integers(-12, 13, size=ROW_SIZE, dtype=np.int16)
vector = rng.integers(-10, 11, size=ROW_SIZE, dtype=np.int16)

def check_int8(a, name):
    if np.any(a < -128) or np.any(a > 127):
        raise ValueError(f"{name} contains values outside signed INT8.")

def fss_csb_encode(row):
    """Prune independently in each bank, then encode kth entries together."""
    if len(row) != ROW_SIZE:
        raise ValueError(f"Expected exactly {ROW_SIZE} weights.")

    pruned = np.zeros(ROW_SIZE, dtype=np.int16)
    bank_kept_values = []
    bank_kept_indices = []

    for b in range(NUM_BANKS):
        bank = row[b * BANK_SIZE:(b + 1) * BANK_SIZE]

        # Select largest magnitudes. Stable sort makes tie handling repeatable.
        selected = np.argsort(-np.abs(bank), kind="stable")[:KEEP_PER_BANK]
        selected = np.sort(selected)  # preserve ascending local positions

        bank_kept_values.append(bank[selected])
        bank_kept_indices.append(selected)

        pruned[b * BANK_SIZE + selected] = bank[selected]

    # Shape: [group, bank, lane-data] -> [4 groups, 4 banks]
    csb_weights = np.zeros((KEEP_PER_BANK, NUM_BANKS), dtype=np.int16)
    csb_indices = np.zeros((KEEP_PER_BANK, NUM_BANKS), dtype=np.uint8)

    for k in range(KEEP_PER_BANK):
        for b in range(NUM_BANKS):
            csb_weights[k, b] = bank_kept_values[b][k]
            csb_indices[k, b] = bank_kept_indices[b][k]

    return pruned, csb_weights, csb_indices

def to_hex8(value):
    # 8-bit two's-complement representation; e.g. -1 -> FF.
    return f"{int(value) & 0xFF:02X}"

def write_mem(filename, values):
    with open(OUT / filename, "w") as f:
        for value in np.asarray(values).reshape(-1):
            f.write(to_hex8(value) + "\n")

check_int8(weights, "weights")
check_int8(vector, "vector")

pruned_weights, csb_weights, csb_indices = fss_csb_encode(weights)

# Exact integer reference. Python integers avoid intermediate overflow.
expected = sum(int(w) * int(x) for w, x in zip(pruned_weights, vector))

# Files consumed by the SystemVerilog testbench.
write_mem("vector.mem", vector)
write_mem("pruned_weights.mem", pruned_weights)
write_mem("csb_weights.mem", csb_weights)
write_mem("csb_indices.mem", csb_indices)

with open(OUT / "expected_result.txt", "w") as f:
    f.write(str(expected) + "\n")

with open(OUT / "reference_report.txt", "w") as f:
    f.write("Original weights:\n")
    f.write(np.array2string(weights) + "\n\n")
    f.write("Pruned weights:\n")
    f.write(np.array2string(pruned_weights) + "\n\n")
    f.write("Vector:\n")
    f.write(np.array2string(vector) + "\n\n")
    f.write("CSB weights [group][bank]:\n")
    f.write(np.array2string(csb_weights) + "\n\n")
    f.write("CSB local indices [group][bank]:\n")
    f.write(np.array2string(csb_indices) + "\n\n")
    f.write(f"Expected dot product: {expected}\n")

print("=== Python reference generation complete ===")
print("Original weights :", weights.tolist())
print("Pruned weights   :", pruned_weights.tolist())
print("Vector           :", vector.tolist())
print("CSB weights:\n", csb_weights)
print("CSB indices:\n", csb_indices)
print("Expected result  :", expected)
print("Generated files in:", OUT)


=== Python reference generation complete ===
Original weights : [7, 6, 8, 9, 5, -10, 1, -8, 2, -1, -4, 1, -3, 7, 0, -3, -1, -12, -2, -7, 11, -1, -3, -3, -9, 10, -8, -2, 2, 3, 12, 3]
Pruned weights   : [0, 0, 8, 9, 0, -10, 0, -8, 0, 0, -4, 0, -3, 7, 0, -3, 0, -12, 0, -7, 11, 0, -3, 0, -9, 10, -8, 0, 0, 0, 12, 0]
Vector           : [0, -9, 6, 5, -3, -9, -10, -10, 9, 2, 7, -5, 2, -3, -4, 2, 10, -7, 6, -9, -5, 4, -1, 10, -2, 4, 3, 7, 5, -2, -5, -10]
CSB weights:
 [[  8  -4 -12  -9]
 [  9  -3  -7  10]
 [-10   7  11  -8]
 [ -8  -3  -3  12]]
CSB indices:
 [[2 2 1 0]
 [3 4 3 1]
 [5 5 4 2]
 [7 7 6 6]]
Expected result  : 271
Generated files in: /content/data
